# 20M reversible LLM — 50M tokens on FineWeb-Edu (baseline vs reversible)

Same code/configs as the local experiment: baseline / rev_mid (leapfrog midpoint) /
rev_mid at max batch. fp16 autocast + GradScaler if bf16 is unavailable (T4).

In [ ]:
!nvidia-smi
import torch
torch.backends.cuda.matmul.allow_tf32 = True
print('torch', torch.__version__, '| bf16 supported:', torch.cuda.is_bf16_supported())

In [ ]:
%%writefile model.py
"""
20M-class GPT with three activation-memory strategies, all with matched parameter counts:

  - "baseline":  standard pre-norm transformer, autograd stores every activation.
  - "rev_euler": two-stream reversible network (Gomez et al. 2017). The coupled update
                 (a,b) -> (a + f(b), b + g(a + f(b))) is a semi-implicit *Euler* step of a
                 2D ODE; its inverse is exact, so only the final pair is stored.
  - "rev_mid":   single-stream *explicit midpoint* (leapfrog / Verlet) reversible stack:
                 x_{i+1} = x_{i-1} + f_i(x_i). Symmetric, 2nd-order; exact inverse
                 x_{i-1} = x_{i+1} - f_i(x_i) from adjacent activation pairs. Full-width
                 blocks (no channel halving).

All variants: d_model=512, 6 layers (blocks), 8 heads, MLP 4x, RoPE, tied embeddings,
vocab 8192 -> 18.87M non-embedding params + 4.19M embedding = 23.06M total.

Streams/residual are kept in fp32; matmuns run under bf16 autocast.
A fused chunked cross-entropy keeps logits memory O(chunk) in fwd and bwd.
"""

import math
from functools import partial

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import custom_fwd, custom_bwd



# ------------------------------------------------------------------- autocast plumbing
# A reversible backward recomputes each block inside torch.autograd.grad. Under autocast
# the bf16 copy of every weight is CACHED, and because the forward ran under no_grad the
# cached copy carries no grad_fn -- the recompute then cannot reach the fp32 parameter and
# autograd.grad(..., allow_unused=True) silently returns None for every Linear weight.
# The model still trains (LayerNorm gains and the embedding are reached directly), so the
# failure is invisible in the loss curve. Re-entering autocast with cache_enabled=False
# inside the backward forces a fresh, differentiable cast.

def _amp_state():
    return (torch.is_autocast_enabled("cuda"), torch.get_autocast_dtype("cuda"))


def _amp_ctx(state):
    enabled, dtype = state
    return torch.autocast("cuda", dtype=dtype, enabled=enabled, cache_enabled=False)


def _require_all_grads(grads, params):
    if any(g is None for g in grads):
        missing = sum(g is None for g in grads)
        raise RuntimeError(
            f"reversible backward produced no gradient for {missing}/{len(params)} block "
            "parameters; the autocast weight cache has detached them from the graph")


# ----------------------------------------------------------------------------- blocks

def rope_cos_sin(T, head_dim, device, base=10000.0):
    half = head_dim // 2
    inv_freq = base ** (-torch.arange(half, device=device, dtype=torch.float32) / half)
    t = torch.arange(T, device=device, dtype=torch.float32)
    freqs = torch.outer(t, inv_freq)                      # (T, half)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    # x: (B, T, H, D). Rotate-half convention on the last dim.
    D = x.shape[-1]
    x1, x2 = x[..., : D // 2], x[..., D // 2:]
    c = cos[None, :, None, :]
    s = sin[None, :, None, :]
    return torch.cat([x1 * c - x2 * s, x2 * c + x1 * s], dim=-1)


class Attention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.hd = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3 * self.h, self.hd).chunk(3, dim=2)
        q = apply_rope(q, cos, sin)
        k = apply_rope(k, cos, sin)
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))   # (B, H, T, hd)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).reshape(B, T, C)
        return self.proj(y)


class MLP(nn.Module):
    def __init__(self, d_model, mult=4):
        super().__init__()
        self.fc = nn.Linear(d_model, mult * d_model, bias=False)
        self.proj = nn.Linear(mult * d_model, d_model, bias=False)

    def forward(self, x):
        return self.proj(F.gelu(self.fc(x)))


class Block(nn.Module):
    """Standard pre-norm block (used by baseline and rev_mid)."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model, bias=False)
        self.attn = Attention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model, bias=False)
        self.mlp = MLP(d_model)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.ln1(x), cos, sin)
        x = x + self.mlp(self.ln2(x))
        return x


class RevBlock(nn.Module):
    """Two-stream reversible block: f = attention on stream b, g = MLP on stream a."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model, bias=False)
        self.attn = Attention(d_model, n_heads)
        self.ln2 = nn.LayerNorm(d_model, bias=False)
        self.mlp = MLP(d_model)

    def f(self, b, cos, sin):
        return self.attn(self.ln1(b), cos, sin)

    def g(self, a, cos, sin):
        return self.mlp(self.ln2(a))


# ------------------------------------------------------------- fused chunked cross-entropy

class FusedCE(torch.autograd.Function):
    """Chunked cross-entropy over flattened tokens; logits never stored for backward.
    forward: loss = mean CE. backward: dlogits = (softmax - onehot) * scale per chunk,
    accumulated into dh and dW with bf16 matmuls."""

    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, h, weight, targets, chunk=8192):
        # h: (N, d) fp32, weight: (V, d), targets: (N,) int64
        ctx.chunk = chunk
        ctx.save_for_backward(h, weight, targets)
        N = h.shape[0]
        with torch.no_grad():
            loss = h.new_zeros((), dtype=torch.float32)
            for i in range(0, N, chunk):
                hc, tc = h[i : i + chunk], targets[i : i + chunk]
                logits = F.linear(hc, weight).float()
                loss = loss + F.cross_entropy(logits, tc, reduction="sum")
        return loss / N

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, grad_out):
        h, weight, targets = ctx.saved_tensors
        N, d = h.shape
        chunk = ctx.chunk
        dh = torch.zeros_like(h)
        dW = torch.zeros_like(weight)
        scale = grad_out / N
        for i in range(0, N, chunk):
            hc, tc = h[i : i + chunk], targets[i : i + chunk]
            logits = F.linear(hc, weight).float()
            p = torch.softmax(logits, dim=-1)
            p[torch.arange(tc.shape[0], device=h.device), tc] -= 1.0
            p = p * scale
            dW += p.T @ hc                       # TF32 matmul (enabled in train.py)
            dh[i : i + chunk] = p @ weight
        return dh, dW, None, None


def lm_loss(h, emb_weight, targets, chunk=8192):
    """h: (B, T, d) -> scalar CE loss."""
    B, T, d = h.shape
    return FusedCE.apply(h.reshape(B * T, d), emb_weight, targets.reshape(B * T), chunk)


# ------------------------------------------------------------------------- reversible stacks

class RevEulerFn(torch.autograd.Function):
    """Two-stream reversible stack. Stores only the final (a, b); backward reconstructs
    the chain top-down and recomputes each block with grad to produce exact gradients."""

    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, a0, b0, blocks, cos, sin, exact_seed=False):
        ctx.blocks = blocks
        ctx.cos, ctx.sin = cos, sin
        ctx.amp = _amp_state()
        ctx.exact_seed = exact_seed
        ctx.seed = (a0, b0) if exact_seed else None
        with torch.no_grad():
            a, b = a0, b0
            for blk in blocks:
                a = a + blk.f(b, cos, sin)
                b = b + blk.g(a, cos, sin)   # g sees the UPDATED a (sequential coupling -> invertible)
        ctx.save_for_backward(a, b)
        return a, b

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, da, db):
        blocks, cos, sin = ctx.blocks, ctx.cos, ctx.sin
        a, b = ctx.saved_tensors
        for i, blk in list(enumerate(blocks))[::-1]:
            if ctx.exact_seed and i == 0:
                a_prev, b_prev = ctx.seed          # exact (a_0, b_0); no cancellation
            else:
                with _amp_ctx(ctx.amp), torch.no_grad():
                    b_prev = b - blk.g(a, cos, sin)
                    a_prev = a - blk.f(b_prev, cos, sin)
            a_prev = a_prev.detach().requires_grad_(True)
            b_prev = b_prev.detach().requires_grad_(True)
            params = [p for p in blk.parameters() if p.requires_grad]
            with _amp_ctx(ctx.amp), torch.enable_grad():
                a_new = a_prev + blk.f(b_prev, cos, sin)
                b_new = b_prev + blk.g(a_new, cos, sin)
            grads = torch.autograd.grad(
                (a_new, b_new), [a_prev, b_prev] + params,
                grad_outputs=(da, db), allow_unused=True,
            )
            _require_all_grads(grads[2:], params)
            da, db = grads[0], grads[1]
            for p, g in zip(params, grads[2:]):
                if g is not None:
                    p.grad = g if p.grad is None else p.grad + g
            a, b = a_prev.detach(), b_prev.detach()
        return da, db, None, None, None, None


class LeapfrogFn(torch.autograd.Function):
    """Single-stream explicit-midpoint (leapfrog) reversible stack:
    x_{i+1} = x_{i-1} + f_i(x_i), x_{-1} = 0. Stores only (x_{N-1}, x_N)."""

    @staticmethod
    @custom_fwd(device_type="cuda")
    def forward(ctx, x0, blocks, cos, sin, exact_seed=False):
        ctx.blocks = blocks
        ctx.cos, ctx.sin = cos, sin
        ctx.amp = _amp_state()
        ctx.exact_seed = exact_seed
        with torch.no_grad():
            xm1 = torch.zeros_like(x0)
            x = x0
            for blk in blocks:
                xm1, x = x, xm1 + blk(x, cos, sin)
        ctx.save_for_backward(*((xm1, x, x0) if exact_seed else (xm1, x)))
        return xm1, x

    @staticmethod
    @custom_bwd(device_type="cuda")
    def backward(ctx, dxm1_top, dx_top):
        blocks, cos, sin = ctx.blocks, ctx.cos, ctx.sin
        saved = ctx.saved_tensors            # (x_{N-1}, x_N[, x_0])
        xi, xi1 = saved[0], saved[1]
        x0_exact = saved[2] if ctx.exact_seed else None
        gxi, gxi1 = dxm1_top, dx_top         # their total grads from above
        for i, blk in list(enumerate(blocks))[::-1]:
            if x0_exact is not None and i == 0:
                # Recovering x_0 by subtraction is catastrophically ill-conditioned: it is
                # embedding-scale while x_1..x_N carry the accumulated residual. x_0 is an
                # input to this Function, so use it directly and skip the subtraction.
                xi = x0_exact
                xim1 = torch.zeros_like(xi)
            else:
                with _amp_ctx(ctx.amp), torch.no_grad():
                    xim1 = xi1 - blk(xi, cos, sin)   # x_{i-1} = x_{i+1} - f_i(x_i)
            xim1 = xim1.detach().requires_grad_(True)
            xi_leaf = xi.detach().requires_grad_(True)
            params = [p for p in blk.parameters() if p.requires_grad]
            with _amp_ctx(ctx.amp), torch.enable_grad():
                xi1_new = xim1 + blk(xi_leaf, cos, sin)
            grads = torch.autograd.grad(
                (xi1_new,), [xim1, xi_leaf] + params,
                grad_outputs=(gxi1,), allow_unused=True,
            )
            _require_all_grads(grads[2:], params)
            # grads[0]: identity path to x_{i-1}; grads[1]: path through f_i into x_i
            new_gxim1 = grads[0]
            new_gxi = gxi + grads[1]
            for p, g in zip(params, grads[2:]):
                if g is not None:
                    p.grad = g if p.grad is None else p.grad + g
            xi, xi1 = xim1.detach(), xi.detach()
            gxi, gxi1 = new_gxim1, new_gxi
        # loop ends with (x_{-1}, x_0): gradient w.r.t. x0 is gxi1; x_{-1} was a constant zero
        return gxi1, None, None, None, None


# --------------------------------------------------------------------------------- models

class GPTBaseline(nn.Module):
    def __init__(self, vocab, d_model=512, n_layers=6, n_heads=8, seq=512):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model, bias=False)
        self.seq = seq
        cos, sin = rope_cos_sin(seq, d_model // n_heads, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(self._init)
        # scale residual output projections for stable depth
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=0.02 / math.sqrt(2 * n_layers))
            nn.init.normal_(blk.mlp.proj.weight, std=0.02 / math.sqrt(2 * n_layers))

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None, ce_chunk=8192):
        cos = self.cos.to(idx.device)[: idx.shape[1]]
        sin = self.sin.to(idx.device)[: idx.shape[1]]
        x = self.emb(idx)
        for blk in self.blocks:
            x = blk(x, cos, sin)
        x = self.ln_f(x)
        if targets is None:
            return x
        return lm_loss(x, self.emb.weight, targets, ce_chunk)


class GPTRevEuler(nn.Module):
    """Two-stream reversible GPT. Each stream is d_model wide; params per block match baseline."""
    exact_seed = False

    def __init__(self, vocab, d_model=512, n_layers=6, n_heads=8, seq=512):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([RevBlock(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model, bias=False)
        self.seq = seq
        cos, sin = rope_cos_sin(seq, d_model // n_heads, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(GPTBaseline._init)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=0.02 / math.sqrt(2 * n_layers))
            nn.init.normal_(blk.mlp.proj.weight, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx, targets=None, ce_chunk=8192):
        cos = self.cos.to(idx.device)[: idx.shape[1]]
        sin = self.sin.to(idx.device)[: idx.shape[1]]
        e = self.emb(idx)
        a, b = RevEulerFn.apply(e, e, self.blocks, cos, sin, self.exact_seed)
        x = self.ln_f(a + b)
        if targets is None:
            return x
        return lm_loss(x, self.emb.weight, targets, ce_chunk)


class GPTRevMid(nn.Module):
    """Single-stream leapfrog (explicit midpoint) reversible GPT; blocks identical to baseline."""
    exact_seed = False

    def __init__(self, vocab, d_model=512, n_layers=6, n_heads=8, seq=512):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([Block(d_model, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model, bias=False)
        self.seq = seq
        cos, sin = rope_cos_sin(seq, d_model // n_heads, "cpu")
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.apply(GPTBaseline._init)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.proj.weight, std=0.02 / math.sqrt(2 * n_layers))
            nn.init.normal_(blk.mlp.proj.weight, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx, targets=None, ce_chunk=8192):
        cos = self.cos.to(idx.device)[: idx.shape[1]]
        sin = self.sin.to(idx.device)[: idx.shape[1]]
        e = self.emb(idx)
        x_prev, x = LeapfrogFn.apply(e, self.blocks, cos, sin, self.exact_seed)
        out = self.ln_f(x_prev + x)
        if targets is None:
            return out
        return lm_loss(out, self.emb.weight, targets, ce_chunk)


class GPTRevMidSeed(GPTRevMid):
    """rev_mid, but the backward takes x_0 from the saved input instead of reconstructing
    it. Costs one extra stored activation; removes the block-0 gradient corruption."""
    exact_seed = True


class GPTRevEulerSeed(GPTRevEuler):
    """rev_euler with the same exact-seed treatment."""
    exact_seed = True


ARCHS = {"baseline": GPTBaseline, "rev_euler": GPTRevEuler, "rev_mid": GPTRevMid,
         "rev_mid_seed": GPTRevMidSeed, "rev_euler_seed": GPTRevEulerSeed}


def count_params(model):
    total = sum(p.numel() for p in model.parameters())
    emb = model.emb.weight.numel()
    return {"total": total, "embedding": emb, "non_embedding": total - emb}


In [ ]:
%%writefile train.py
"""Trainer for the 20M-class reversible-vs-baseline experiments.

Examples:
  python train.py --arch baseline --batch 16 --tokens 50e6 --out runs/baseline_b16
  python train.py --arch rev_mid  --batch 16 --tokens 50e6 --out runs/revmid_b16
  python train.py --arch rev_mid  --batch 96 --tokens 50e6 --out runs/revmid_bmax --lr 4.2e-3
  python train.py --arch rev_mid  --probe-batch 128          # 3-step memory probe
"""

import argparse
import json
import math
import subprocess
import threading
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

from model import ARCHS, count_params

ROOT = Path(__file__).parent
DATA = ROOT / "data"
VOCAB = 8192


def get_batch(data, idxs, seq, device):
    xs = np.stack([data[i : i + seq] for i in idxs])
    ys = np.stack([data[i + 1 : i + 1 + seq] for i in idxs])
    return (torch.from_numpy(xs.astype(np.int64)).to(device, non_blocking=True),
            torch.from_numpy(ys.astype(np.int64)).to(device, non_blocking=True))


@torch.no_grad()
def evaluate(model, val_data, seq, device, max_tokens=2_000_000, batch=8):
    model.eval()
    n = len(val_data) - seq - 1
    rng = np.random.default_rng(7)
    total, count = 0.0, 0
    done = 0
    while done < max_tokens:
        b = min(batch, (max_tokens - done) // seq)
        if b <= 0:
            break
        idxs = rng.integers(0, n, size=b)
        x, y = get_batch(val_data, idxs, seq, device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(x, y)
        total += loss.item() * b * seq
        count += b * seq
        done += b * seq
    model.train()
    return total / count


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--arch", required=True, choices=list(ARCHS))
    ap.add_argument("--batch", type=int, default=16)
    ap.add_argument("--seq", type=int, default=512)
    ap.add_argument("--tokens", type=float, default=50e6)
    ap.add_argument("--lr", type=float, default=3e-3)
    ap.add_argument("--warmup-frac", type=float, default=0.02)
    ap.add_argument("--seed", type=int, default=1234)
    ap.add_argument("--out", default=None)
    ap.add_argument("--log-every", type=int, default=50)
    ap.add_argument("--ce-chunk", type=int, default=8192)
    ap.add_argument("--probe-batch", type=int, default=0, help="run 3 steps at this batch, report memory, exit")
    ap.add_argument("--save-ckpt", action="store_true")
    ap.add_argument("--thermal-guard", action="store_true",
                    help="stop the run when the GPU enters the throttled regime "
                         "(SW thermal slowdown flag active, or >=87C)")
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    device = "cuda"

    global AMP_DTYPE, SCALER
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    SCALER = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
    print(f"autocast dtype: {AMP_DTYPE}", flush=True)
    out = Path(args.out) if args.out else ROOT / "runs" / f"{args.arch}_b{args.batch}"
    out.mkdir(parents=True, exist_ok=True)

    train_mm = np.memmap(DATA / "train.bin", dtype=np.uint16, mode="r")
    val_mm = np.memmap(DATA / "val.bin", dtype=np.uint16, mode="r")

    model = ARCHS[args.arch](VOCAB, seq=args.seq).to(device)
    pcount = count_params(model)
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        (no_decay if p.ndim < 2 else decay).append(p)
    opt = torch.optim.AdamW(
        [{"params": decay, "weight_decay": 0.1}, {"params": no_decay, "weight_decay": 0.0}],
        lr=args.lr, betas=(0.9, 0.95), eps=1e-8, fused=True)

    steps = int(args.tokens // (args.batch * args.seq))
    warmup = max(30, int(steps * args.warmup_frac))

    if args.probe_batch:
        args.batch = args.probe_batch
        steps = 3

    print(f"arch={args.arch} params={pcount} batch={args.batch} seq={args.seq} steps={steps} lr={args.lr}", flush=True)

    g = torch.Generator().manual_seed(args.seed)
    n_train = len(train_mm) - args.seq - 1

    guard_stop = threading.Event()
    guard_done = threading.Event()
    guard_state = {"reason": None}
    guard_samples = []

    def _guard_smi():
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=temperature.gpu,clocks.sm,power.draw,"
             "clocks_event_reasons.sw_thermal_slowdown", "--format=csv,noheader,nounits"],
            capture_output=True, text=True).stdout.strip()
        p = [v.strip() for v in out.split(",")]
        return float(p[0]), float(p[1]), float(p[2]), p[3] == "Active"

    def thermal_watchdog():
        hot, t0 = 0, time.time()
        while not guard_done.is_set():
            try:
                temp, clk, pwr, sw = _guard_smi()
            except Exception:
                time.sleep(2.0)
                continue
            guard_samples.append({"t": round(time.time() - t0, 1), "temp": temp,
                                  "clk": clk, "pwr": pwr, "sw_thermal": sw})
            # the documented throttled regime: SW thermal governor engaged (its flag,
            # or >=87C) at ANY clock level; require 2 consecutive hits
            throttled = temp >= 87 or sw
            hot = hot + 1 if throttled else 0
            if hot >= 2:
                guard_state["reason"] = (f"throttled: {temp:.0f}C {clk:.0f}MHz "
                                         f"sw_thermal_slowdown={sw}")
                guard_stop.set()
                return
            time.sleep(2.0)

    if args.thermal_guard:
        threading.Thread(target=thermal_watchdog, daemon=True).start()

    curve, mem_peak_alloc, mem_peak_res = [], 0, 0
    t_start = time.time()
    t_steady_start, tokens_steady = None, 0
    ema = None
    steps_done = steps

    for step in range(steps):
        lr = args.lr * min((step + 1) / warmup, 1.0) if step < warmup else \
             args.lr * (0.1 + 0.45 * (1 + math.cos(math.pi * (step - warmup) / (steps - warmup))))
        for group in opt.param_groups:
            group["lr"] = lr

        idxs = torch.randint(0, n_train, (args.batch,), generator=g).numpy()
        x, y = get_batch(train_mm, idxs, args.seq, device)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            loss = model(x, y, ce_chunk=args.ce_chunk)
        if SCALER.is_enabled():
            SCALER.scale(loss).backward()
            SCALER.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            SCALER.step(opt)
            SCALER.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        opt.zero_grad(set_to_none=True)

        lv = loss.item()
        ema = lv if ema is None else 0.95 * ema + 0.05 * lv
        curve.append((step, lv, lr))

        if step == 5:  # start steady-state timing after warm-up of allocator
            torch.cuda.synchronize()
            t_steady_start = time.time()
            tokens_steady = 0
        if step >= 5:
            tokens_steady += args.batch * args.seq

        mem_peak_alloc = max(mem_peak_alloc, torch.cuda.max_memory_allocated())
        mem_peak_res = max(mem_peak_res, torch.cuda.max_memory_reserved())

        if (step + 1) % args.log_every == 0 or step == steps - 1:
            el = time.time() - t_start
            tps = tokens_steady / max(time.time() - t_steady_start, 1e-9) if step >= 5 else 0
            print(f"step {step+1}/{steps} loss {lv:.4f} ema {ema:.4f} lr {lr:.2e} "
                  f"tok/s {tps/1e3:.1f}k elapsed {el:.0f}s", flush=True)

        if args.thermal_guard and guard_stop.is_set():
            steps_done = step + 1
            print(f"THERMAL GUARD: stopping at step {step+1}/{steps}: {guard_state['reason']}",
                  flush=True)
            break

    guard_done.set()
    wall = time.time() - t_start
    tps_steady = tokens_steady / max(time.time() - t_steady_start, 1e-9) if t_steady_start else 0.0
    stopped = args.thermal_guard and guard_state["reason"] is not None

    if stopped:
        val_loss, val_ppl = None, None
    else:
        val_loss = evaluate(model, val_mm, args.seq, device)
        val_ppl = math.exp(val_loss)
    metrics = {
        "arch": args.arch, "batch": args.batch, "seq": args.seq,
        "tokens_planned": steps * args.batch * args.seq, "steps_planned": steps,
        "steps_done": steps_done, "tokens_done": steps_done * args.batch * args.seq,
        "tokens": steps_done * args.batch * args.seq, "steps": steps_done,
        "lr": args.lr, "params": pcount,
        "final_train_loss_ema": ema,
        "final_val_loss": val_loss, "val_ppl": val_ppl,
        "tokens_per_s": tps_steady,
        "peak_mem_allocated_mb": mem_peak_alloc / 2**20,
        "peak_mem_reserved_mb": mem_peak_res / 2**20,
        "wall_s": wall, "gpu": torch.cuda.get_device_name(0),
        "torch": torch.__version__, "seed": args.seed,
        "stopped_reason": guard_state["reason"],
        "guard_samples_n": len(guard_samples),
    }
    (out / "metrics.json").write_text(json.dumps(metrics, indent=2))
    np.save(out / "curve.npy", np.array(curve))
    if args.thermal_guard:
        (out / "guard.json").write_text(json.dumps(
            {"reason": guard_state["reason"], "samples": guard_samples}, indent=1))
    if args.save_ckpt:
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "step": steps}, out / "ckpt.pt")
    print(json.dumps(metrics, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile data_prep.py
"""Prepare data: stream a FineWeb-Edu parquet shard, train an 8k BPE tokenizer,
write ~57M tokens as uint16 binaries (train.bin / val.bin)."""

import io
import sys
import time
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

ROOT = Path(__file__).parent
DATA = ROOT / "data"
PARQUET = DATA / "fineweb-edu-00000.parquet"
VOCAB = 8192
TRAIN_TOKENS = 52_000_000
VAL_TOKENS = 5_000_000
TOTAL_NEEDED = TRAIN_TOKENS + VAL_TOKENS


def main():
    t0 = time.time()
    pf = pq.ParquetFile(PARQUET)
    print(f"parquet: {pf.metadata.num_rows} rows, {pf.metadata.num_row_groups} row groups")

    # ---- collect a text sample for tokenizer training (~25MB)
    sample, sample_chars = [], 0
    texts_iter = (batch.column("text").to_pylist()
                  for batch in pf.iter_batches(batch_size=2048, columns=["text"]))
    for batch_texts in texts_iter:
        for t in batch_texts:
            if len(t) >= 300:
                sample.append(t)
                sample_chars += len(t)
        if sample_chars > 25_000_000:
            break
    print(f"tokenizer sample: {len(sample)} docs, {sample_chars/1e6:.1f} MB")

    tok = Tokenizer(models.BPE(unk_token=None))
    tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    trainer = trainers.BpeTrainer(
        vocab_size=VOCAB,
        special_tokens=["<|endoftext|>"],
        show_progress=False,
    )
    tok.train_from_iterator(sample, trainer)
    eot = tok.token_to_id("<|endoftext|>")
    assert eot == 0, f"EOT id is {eot}, expected 0"
    tok.save(str(DATA / "bpe_8192.json"))
    print(f"tokenizer trained: vocab {tok.get_vocab_size()} ({time.time()-t0:.0f}s)")

    del sample

    # ---- tokenize the corpus into a uint16 buffer
    buf = np.empty(TOTAL_NEEDED + 1_000_000, dtype=np.uint16)
    n = 0
    n_docs = 0
    for batch_texts in texts_iter:
        texts = [t for t in batch_texts if len(t) >= 300]
        if not texts:
            continue
        encs = tok.encode_batch(texts, add_special_tokens=False)
        for e in encs:
            ids = e.ids
            if n + len(ids) + 1 > len(buf):
                break
            buf[n : n + len(ids)] = ids
            n += len(ids)
            buf[n] = eot
            n += 1
            n_docs += 1
        if n % 10_000_000 < 40_000:
            print(f"  {n/1e6:.1f}M tokens ({time.time()-t0:.0f}s)", flush=True)
        if n >= TOTAL_NEEDED:
            break

    print(f"tokenized {n/1e6:.1f}M tokens from {n_docs} docs ({time.time()-t0:.0f}s)")
    train = buf[:TRAIN_TOKENS]
    val = buf[TRAIN_TOKENS : TRAIN_TOKENS + VAL_TOKENS]
    train.tofile(DATA / "train.bin")
    val.tofile(DATA / "val.bin")
    print(f"train.bin {TRAIN_TOKENS/1e6:.0f}M tokens, val.bin {VAL_TOKENS/1e6:.0f}M tokens")


if __name__ == "__main__":
    main()


## Fetch the corpus
`data_prep.py` reads a local parquet shard, which does not exist on a fresh Colab runtime. This pulls the same FineWeb-Edu shard used locally (~2.1 GB, a few minutes) and puts it where `data_prep.py` expects it.

In [ ]:

!pip -q install tokenizers huggingface_hub

import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs("data", exist_ok=True)
dest = "data/fineweb-edu-00000.parquet"
if not os.path.exists(dest):
    p = hf_hub_download("HuggingFaceFW/fineweb-edu", "sample/10BT/000_00000.parquet",
                        repo_type="dataset")
    try:                      # symlink to avoid a second 2.1 GB on disk
        os.symlink(p, dest)
    except OSError:
        shutil.copy(p, dest)
print(f"{dest}: {os.path.getsize(dest)/1e9:.2f} GB")


In [ ]:
!python data_prep.py

In [ ]:
%%writefile gradcheck.py
"""
Gradient-correctness verification: custom reversible Functions and FusedCE must produce
the same gradients as a naive store-everything autograd reference. Run in fp32 on CPU
(SDPA works on CPU too) so any mismatch is numerical, not hardware.
"""

import torch
import torch.nn.functional as F

from model import GPTBaseline, GPTRevEuler, GPTRevMid, FusedCE, lm_loss

torch.manual_seed(0)
V, D, L, H, T, B = 256, 64, 4, 4, 32, 2


def copy_state(src, dst):
    dst.load_state_dict(src.state_dict())


def param_grads(model):
    return {k: p.grad.clone() for k, p in model.named_parameters()}


def rel_err(a, b):
    denom = b.abs().max().clamp_min(1e-12)
    return (a - b).abs().max() / denom


def check_reversible(RevCls, name, top="sum"):
    base = GPTBaseline(V, D, L, H, T)
    rev = RevCls(V, D, L, H, T)
    copy_state(base, rev)

    idx = torch.randint(0, V, (B, T))
    tgt = torch.randint(0, V, (B, T))

    # reference: same architecture run with plain autograd (store everything)
    ref = RevCls(V, D, L, H, T)
    copy_state(base, ref)
    x = ref.emb(idx)
    cos, sin = ref.cos, ref.sin
    if name == "rev_euler":
        a, b = x, x
        for blk in ref.blocks:
            a = a + blk.f(b, cos, sin)
            b = b + blk.g(a, cos, sin)
        h = ref.ln_f(a + b)
    else:  # leapfrog
        xm1 = torch.zeros_like(x)
        for blk in ref.blocks:
            xm1, x = x, xm1 + blk(x, cos, sin)
        h = ref.ln_f(xm1 + x)
    h_ref = ref.ln_f(a + b) if name == "rev_euler" else None
    if name != "rev_euler":
        h_ref = h
    logits_ref = F.linear(h_ref.reshape(-1, D), ref.emb.weight).float()
    loss_ref = F.cross_entropy(logits_ref, tgt.reshape(-1))
    loss_ref.backward()
    ref_grads = param_grads(ref)

    # custom reversible path
    rev.zero_grad(set_to_none=True)
    loss_rev = rev(idx, tgt)
    loss_rev.backward()
    rev_grads = param_grads(rev)

    worst = 0.0
    for k in ref_grads:
        e = rel_err(rev_grads[k], ref_grads[k])
        worst = max(worst, e.item())
    print(f"[{name}] loss_ref={loss_ref.item():.6f} loss_rev={loss_rev.item():.6f} "
          f"d_loss={abs(loss_ref - loss_rev).item():.2e}  max_rel_grad_err={worst:.2e}")
    assert worst < 1e-5, f"{name}: gradient mismatch {worst}"
    return worst


def check_fused_ce():
    torch.manual_seed(1)
    N, d, V2 = 1000, 64, 512
    h = torch.randn(N, d, requires_grad=True)
    w = torch.randn(V2, d, requires_grad=True)
    t = torch.randint(0, V2, (N,))

    # reference
    h2 = h.detach().clone().requires_grad_(True)
    w2 = w.detach().clone().requires_grad_(True)
    loss_ref = F.cross_entropy(F.linear(h2, w2).float(), t)
    loss_ref.backward()

    loss_fused = FusedCE.apply(h, w, t, 256)
    loss_fused.backward()

    dl = abs(loss_ref.item() - loss_fused.item()) / loss_ref.item()
    eh = rel_err(h.grad, h2.grad).item()
    ew = rel_err(w.grad, w2.grad).item()
    print(f"[FusedCE] rel_loss_err={dl:.2e} dH_err={eh:.2e} dW_err={ew:.2e}")
    assert dl < 1e-5 and eh < 1e-4 and ew < 1e-4


if __name__ == "__main__":
    check_fused_ce()
    check_reversible(GPTRevEuler, "rev_euler")
    check_reversible(GPTRevMid, "rev_mid")
    print("ALL GRADIENT CHECKS PASSED")


In [ ]:
!python gradcheck.py

In [ ]:

# Find the largest batch that fits THIS runtime (T4/L4/A100 differ a lot).
# --probe-batch ends by dumping metrics JSON, so parse that rather than the last line.
import subprocess, sys, json, re, torch

free, total = torch.cuda.mem_get_info()
print(f"free {free/2**20:.0f} MB / total {total/2**20:.0f} MB\n")

def probe(arch, b):
    r = subprocess.run([sys.executable, "train.py", "--arch", arch,
                        "--probe-batch", str(b)], capture_output=True, text=True)
    m = re.search(r"\{[\s\S]*\}\s*$", r.stdout)
    if r.returncode != 0 or not m:
        err = (r.stderr.strip().splitlines() or ["failed"])[-1]
        print(f"{arch:9s} b={b:4d}: FAILED/OOM  {err[:80]}")
        return False
    d = json.loads(m.group(0))
    print(f"{arch:9s} b={b:4d}: alloc {d['peak_mem_allocated_mb']:7.0f} MB   "
          f"reserved {d['peak_mem_reserved_mb']:7.0f} MB   {d['tokens_per_s']/1e3:6.1f}k tok/s")
    # a batch only really fits if reserved stays inside free VRAM
    return d["peak_mem_reserved_mb"] < free / 2**20

for arch, batches in [("baseline", [64, 128, 256]), ("rev_mid", [256, 384, 512, 640])]:
    print(f"=== {arch} ===")
    for b in batches:
        if not probe(arch, b):
            break


In [ ]:
!python train.py --arch baseline --batch 16 --lr 7.5e-4 --tokens 50e6 --out runs/r1_baseline_b16 --log-every 200

In [ ]:
!python train.py --arch rev_mid --batch 16 --lr 7.5e-4 --tokens 50e6 --out runs/r2_rev_mid_b16 --log-every 200

In [ ]:
!python train.py --arch rev_mid --batch 192 --lr 1.3e-3 --tokens 50e6 --out runs/r3_rev_mid_b192 --log-every 100

In [ ]:
# optional: push to the largest batch that fits this runtime (edit batch to the probe result)
!python train.py --arch rev_mid --batch 512 --lr 2.1e-3 --tokens 50e6 --out runs/r4_rev_mid_bmax --log-every 50

In [ ]:

import json, glob
print(f"{'run':28s} {'arch':9s} {'batch':>5s} {'val_loss':>9s} {'ppl':>8s} {'tok/s':>8s} {'peakMB':>7s}")
for f in sorted(glob.glob("runs/r*/metrics.json")):
    m = json.load(open(f))
    vl = f"{m['final_val_loss']:.4f}" if m.get("final_val_loss") else "n/a"
    pl = f"{m['val_ppl']:.1f}" if m.get("val_ppl") else "n/a"
    print(f"{f.split('/')[1]:28s} {m['arch']:9s} {m['batch']:5d} {vl:>9s} {pl:>8s} "
          f"{m['tokens_per_s']/1e3:7.1f}k {m['peak_mem_allocated_mb']:7.0f}")
